# 05 — Model Training, Validation & Evaluation

This notebook uses a chronological train/test split. The latest 20% of orders are held out for the final evaluation. Preprocessing is inside each scikit-learn Pipeline, so imputers, scaling, and encoding are fitted using training data only.

Run `04_feature_engineering.ipynb` first.

In [1]:

import numpy as np
import pandas as pd
import gc
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from sklearn.base import clone

RANDOM_STATE = 42


df = pd.read_csv("data/processed/orders_features.csv")

y = df["late"].astype(int)

exclude = [
    "late",
    "order_id",
    "customer_id",
    "order_status",
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "delivery_delay_days"
]

X = df.drop(
    columns=[c for c in exclude if c in df.columns]
).copy()

constant_cols = [
    c for c in X.columns
    if X[c].nunique(dropna=False) <= 1
]

X.drop(columns=constant_cols, inplace=True)

print("Dataset:", X.shape)
print("\nTarget distribution:")
print(y.value_counts(normalize=True))


Dataset: (96470, 21)

Target distribution:
late
0    0.918876
1    0.081124
Name: proportion, dtype: float64


C:\Users\Administrator\AppData\Local\Temp\ipykernel_7252\762659486.py:55: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X.drop(columns=constant_cols, inplace=True)


In [2]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)
print("Testing:", X_test.shape)


Training: (67529, 21)
Validation: (14470, 21)
Testing: (14471, 21)


In [3]:
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    c for c in X_train.columns
    if c not in numeric_features
]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=50,
        max_categories=30,
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ],
    remainder="drop",
    sparse_threshold=1.0
)


In [4]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1200,
        class_weight="balanced",
        solver="liblinear",
        C=1.0,
        random_state=42
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=120,
        max_depth=20,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced",
        n_jobs=1,
        random_state=42
    )
}

trained_models = {}
results = []

for name, model in models.items():

    print(f"\nTraining {name}...")

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    trained_models[name] = pipeline

    probabilities = pipeline.predict_proba(X_valid)[:, 1]

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_valid, predictions),
        "Precision": precision_score(
            y_valid, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_valid, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_valid, predictions, zero_division=0
        ),
        "ROC_AUC": roc_auc_score(y_valid, probabilities),
        "Average_Precision": average_precision_score(
            y_valid, probabilities
        )
    })

    gc.collect()

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "Accuracy",
        ascending=False
    )
)



Training Logistic Regression...

Training Extra Trees...


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Average_Precision
1,Extra Trees,0.775605,0.184667,0.517036,0.272136,0.733859,0.223838
0,Logistic Regression,0.567173,0.127070,0.738501,0.216831,0.703288,0.166208


In [5]:
best_model_name = (
    results_df.sort_values(
        ["Accuracy", "F1"],
        ascending=False
    ).iloc[0]["Model"]
)

best_model = trained_models[best_model_name]

valid_probabilities = best_model.predict_proba(
    X_valid
)[:, 1]

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.02):

    predictions = (
        valid_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_valid, predictions),
        "Precision": precision_score(
            y_valid, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_valid, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_valid, predictions, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.sort_values(
        ["Accuracy", "F1"],
        ascending=False
    ).head(10)
)

best_threshold = threshold_df.sort_values(
    ["Accuracy", "F1"],
    ascending=False
).iloc[0]["Threshold"]

print("Best Model:", best_model_name)
print("Best Threshold:", best_threshold)


,Threshold,Accuracy,Precision,Recall,F1
29,0.78,0.918936,0.600000,0.002555,0.005089
30,0.80,0.918867,0.000000,0.000000,0.000000
28,0.76,0.918728,0.437500,0.005963,0.011765
27,0.74,0.917899,0.360000,0.015332,0.029412
26,0.72,0.917761,0.438462,0.048552,0.087423
25,0.70,0.916655,0.433884,0.089438,0.148305
24,0.68,0.913960,0.396501,0.115843,0.179301
23,0.66,0.910574,0.371245,0.147359,0.210976
22,0.64,0.905805,0.346840,0.182283,0.238973
21,0.62,0.897097,0.310924,0.220613,0.258097


Best Model: Extra Trees
Best Threshold: 0.78


In [6]:
test_probabilities = best_model.predict_proba(
    X_test
)[:, 1]

test_predictions = (
    test_probabilities >= best_threshold
).astype(int)

final_results = pd.DataFrame([{
    "Model": best_model_name,
    "Accuracy": accuracy_score(y_test, test_predictions),
    "Precision": precision_score(
        y_test, test_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, test_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_test, test_predictions, zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_test, test_probabilities
    ),
    "Average_Precision": average_precision_score(
        y_test, test_probabilities
    )
}])

display(final_results)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Average_Precision
0,Extra Trees,0.91908,0.714286,0.004259,0.008467,0.72187,0.240866



Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     13297
           1       0.71      0.00      0.01      1174

    accuracy                           0.92     14471
   macro avg       0.82      0.50      0.48     14471
weighted avg       0.90      0.92      0.88     14471


Confusion Matrix:
[[13295     2]
 [ 1169     5]]


In [7]:
from pathlib import Path
import os
import joblib

ROOT = Path.cwd()

if not (ROOT / "models").exists() and (ROOT.parent / "models").exists():
    ROOT = ROOT.parent

MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

model_artifact = {
    "pipeline": best_model,
    "threshold": float(best_threshold),
    "model_name": best_model_name,
    "feature_columns": X_train.columns.tolist()
}

model_path = MODEL_DIR / "late_delivery_pipeline_v2.joblib"

joblib.dump(model_artifact, model_path)

final_results.to_csv(
    REPORT_DIR / "model_v2_test_metrics.csv",
    index=False
)

print("Model saved successfully!")
print("Model location:", model_path)
print("Model exists:", model_path.exists())
print("Model size:", round(model_path.stat().st_size / (1024 * 1024), 2), "MB")


Model saved successfully!
Model location: c:\Users\Administrator\Desktop\olist_notebooks\models\late_delivery_pipeline_v2.joblib
Model exists: True
Model size: 26.96 MB
